# SOL Wallet Intelligence

Explores wallet flow signals joined to SOL price from `mart_signals_daily`.
Runs top-to-bottom with synthetic fallback when wallet data is absent.

In [ ]:
from pathlib import Path

import plotly.graph_objects as go
import polars as pl
from dotenv import load_dotenv

from ccquant.forecasting import load_signals_panel, load_wallet_panel

load_dotenv()
DB = Path("data/ccquant.duckdb")

In [ ]:
if DB.exists():
    signals = load_signals_panel(DB)
    wallet = load_wallet_panel(DB)
else:
    signals = pl.DataFrame(
        {
            "symbol": ["SOL"],
            "date": ["2026-07-01"],
            "close": [150.0],
            "smart_money_netflow_usd": [1000.0],
            "kol_buy_count": [2],
            "cabal_alert_count": [1],
        }
    )
    wallet = pl.DataFrame(
        {
            "date": ["2026-07-01"],
            "chain": ["solana"],
            "smart_money_netflow_usd": [1000.0],
            "kol_buy_count": [2],
            "cabal_alert_count": [1],
        }
    )

sol = signals.filter(pl.col("symbol") == "SOL") if "symbol" in signals.columns else signals
display(sol.tail(5))
display(wallet.tail(5))

In [ ]:
if {"date", "close", "smart_money_netflow_usd"}.issubset(sol.columns):
    plot_df = sol.select(["date", "close", "smart_money_netflow_usd"]).drop_nulls()
    if plot_df.height > 0:
        dates = plot_df["date"].to_list()
        fig = go.Figure()
        fig.add_trace(
            go.Scatter(x=dates, y=plot_df["close"].to_list(), name="close")
        )
        fig.add_trace(
            go.Scatter(
                x=dates,
                y=plot_df["smart_money_netflow_usd"].to_list(),
                name="smart_money_netflow_usd",
            )
        )
        fig.update_layout(title="SOL price vs smart-money netflow")
        fig.show()
    else:
        print("No SOL wallet signal rows yet — run: uv run ccquant sync wallets")
else:
    print("Wallet columns not present — build dbt marts first")